In [1]:
import networkx as nx 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

### script specific functions

In [2]:
import numpy as np  
import random 

def generate_peripheral_position():
    coords = []
    for _ in range(3):
        edge_val = np.random.choice([0.0, 1.0])
        jitter = np.random.uniform(-0.1, 0.1)
        coords.append(np.clip(edge_val + jitter, 0.0, 1.0))
    return tuple(coords)
    
def generate_random_central_position():
    coords = []
    for _ in range(3):
        edge_val = np.random.choice([0.49, 0.51])
        jitter = np.random.uniform(-0.01, 0.01)
        coords.append(np.clip(edge_val + jitter, 0.49,0.51))
    return tuple(coords)

def generate_random_spherical_position(spatial_range=(0.8, 1.0)):
        radius = random.uniform(*spatial_range)
        theta = random.uniform(0, 2 * np.pi)
        phi = random.uniform(0, np.pi)
        x = radius * np.sin(phi) * np.cos(theta)
        y = radius * np.sin(phi) * np.sin(theta)
        z = radius * np.cos(phi)
        
        # Normalize to 0-1 range
        x = (x + 1) / 2
        y = (y + 1) / 2
        z = (z + 1) / 2
        
        return (x, y, z)

### Connect

In [6]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()

✅ Connected to /main


✅ Connected to /main


In [7]:
#client.disconnect()

In [190]:
# FIRST : upload a Graph with all needed information as node annotation attribute 
# (e.g. this can include initial positions in case you want to access them and 
# for time reasons start off with a precalculated layout, node types if there are 
# several graphs merged into one, ... )

# then you run the cells in this notebook one by one. 

# Note: input files are stored locally and not shared via git (temp-files folder)

In [14]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'CDK5'),
 (1, 'CircLadderGraph-xsmall'),
 (2, 'CLGraph_TEST'),
 (3, 'diffusion'),
 (4, 'Exposurome'),
 (5, 'GenExpression_01'),
 (6, 'GenExpression_02'),
 (7, 'imunet_250130-X0-inter'),
 (8, 'imunet_250130-X1-inter'),
 (9, 'Interactive_Project_T01'),
 (10, 'JSON_autocore'),
 (11, 'JSON_barbellgraph'),
 (12, 'JSON_Zachary'),
 (13, 'Microplastics_Human_Health'),
 (14, 'Powergrid_Europe'),
 (15, 'Realtime-project'),
 (16, 'Sphere_Torus'),
 (17, 'Sphere_Torus_Morph'),
 (18, 'Teapot'),
 (19, 'TEAPOT-RealtimeTesting'),
 (20, 'TheMandelbulb_edges')]

In [15]:
# select a project to work with
sel_id = 13
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()

session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)
tools = AnalysisToolkit(session, tex_gen, syncer, file_mgr)

Session Graph loaded from project folder. Data: Nodes: 16193 Links: 13348


In [16]:
latest_message = client.latest_data
session.get_active_layouts(latest_message)

Layout: scene05-BPA-molecule Selected index: 5
Layout: scene05-BPA-molecule Selected index: 5
Layout: scene05-BPA-molecule Selected index: 5


✅ Connected to /main


### Reconstruct Graph from Project

In [17]:
G = session.load_graph_from_project()
print("G_nodes:", len(G.nodes()))
print("G_edges:", len(G.edges()))

Session Graph loaded from project folder. Data: Nodes: 16193 Links: 13348
G_nodes: 16193
G_edges: 13348


In [18]:
# check for node attributes 
node_attr = G.nodes(data=True)

# quick check
node_attr[1]

{'name': 'Liver-N1',
 'attrlist': {'original_id': 1,
  'new_id': 1,
  'name': 'Liver-N1',
  'init_pos': [0.47072015929023164, 0.5383522654236217, 0.501755280716959],
  'type': 'organs'}}

In [19]:
# get attr "init-pos" and set to Graph nodes attributes "pos"

pos = {}
for node in G.nodes():
    pos[node] = node_attr[node]["attrlist"]["init_pos"]

# quick check
pos[0]

[0.539464182615036, 0.5037300173195982, 0.5525095555563215]

In [20]:
nx.set_node_attributes(G, pos, 'pos')

In [21]:
node_type = {}
for node in G.nodes():
    node_type[node] = node_attr[node]["attrlist"]["type"]

# SCENES

### SCENE 1 - unesco world heritage sites + globe 

In [154]:
unesco_nodes = []
for node, attr in node_attr:
    if 'unesco' in node_type[node]:
        unesco_nodes.append(node)

print("unesco_nodes:", len(unesco_nodes))

unesco_nodes: 1155


In [155]:
# node positions 
    
pos_unesco = {}
for i in G.nodes():
    if i in unesco_nodes:
        pos_unesco[i] = G.nodes[i]['pos']
    else:
        pos_unesco[i] = generate_random_spherical_position(spatial_range=(0.0, 0.5))

In [162]:
# node colors 

col_cultural = (167,81,3,80)
col_natural = (91,231,0,200)
col_others = (131,144,70,90)

nodecol_unesco = {}
for node in G.nodes():
    if "Cultural" in node_attr[node]["attrlist"]["type"]:
        nodecol_unesco[node] = col_cultural
    elif "Natural" in node_attr[node]["attrlist"]["type"]:
        nodecol_unesco[node] = col_natural
    else:
        nodecol_unesco[node] = (0,0,0,0)

In [163]:
linkcol_unesco = {} # empty : all edges should be black 

In [161]:
layout_name = "scene01-UNESCO-sites_geo"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_unesco, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_unesco, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_unesco, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 2 - Blood system

In [27]:
# node colors 

col_blood_rgba = (250,84,0,100) 

blood_nodes = []
for node, attr in node_attr:
    if 'BloodSystem-' in attr['name']:
        blood_nodes.append(node)

nodecol_bloodsystem = {}
for i in G.nodes():
    if i in blood_nodes:
       nodecol_bloodsystem[i] = col_blood_rgba
    else:
        nodecol_bloodsystem[i] = (0,0,0,0)

In [28]:
# node positions 

ppi_nodes = []
for node, attr in node_attr:
    if 'ppi' in node_type[node]:
        ppi_nodes.append(node)
        

pos_organs = {}
for i in G.nodes():
    if i in blood_nodes:
        pos_organs[i] = G.nodes[i]['pos']
    elif i in ppi_nodes:
        pos_organs[i] = generate_random_spherical_position()
    else:
        pos_organs[i] = generate_random_central_position()

In [29]:
# show min and max values of x y z of posG_organs
pos_organs_array = np.array(list(pos_organs.values()))
print("min x:", np.min(pos_organs_array[:,0]))
print("max x:", np.max(pos_organs_array[:,0]))
print("min y:", np.min(pos_organs_array[:,1]))
print("max y:", np.max(pos_organs_array[:,1]))
print("min z:", np.min(pos_organs_array[:,2]))
print("max z:", np.max(pos_organs_array[:,2]))

min x: 0.006668065269763379
max x: 0.9946878731123363
min y: 0.005956442848542465
max y: 0.9977300779793271
min z: 0.001025718760185812
max z: 0.9994507780351564


In [30]:
# link colors 

linkcol_bloodsystem = {}

l_blood_edges = []

for i in G.edges():
    if i[0] in blood_nodes and i[1] in blood_nodes:
       l_blood_edges.append(i)
       linkcol_bloodsystem[i] = col_blood_rgba

    else:
        linkcol_bloodsystem[i] = (0,0,0,0)

In [31]:
layout_name = "scene02-Organs-Bloodsystem"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_bloodsystem, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_bloodsystem, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_organs, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 3 - Blood System + Heart 

In [32]:
# node colors 
col_heart_rgba = (228,9,9,100)

heart_nodes = []
for node, attr in node_attr:
    if 'Heart-' in attr['name']:
        heart_nodes.append(node)

nodecol_blood_heart = {}
for i in G.nodes():
    if i in heart_nodes:
        nodecol_blood_heart[i] = col_heart_rgba
    elif i in blood_nodes:
       nodecol_blood_heart[i] = col_blood_rgba
    else:
        nodecol_blood_heart[i] = (0,0,0,0)

In [33]:
pos_organs = {}
for i in G.nodes():
    if i in heart_nodes or i in blood_nodes:
        pos_organs[i] = G.nodes[i]['pos']
    elif i in ppi_nodes:
        pos_organs[i] = generate_random_spherical_position()
    else:
        pos_organs[i] = generate_random_central_position()

In [34]:
# link colors 

linkcol_blood_heart = {}

l_heart_edges = []
l_blood_edges = []

for i in G.edges():
    if i[0] in heart_nodes and i[1] in heart_nodes:
        l_heart_edges.append(i)
        linkcol_blood_heart[i] = col_heart_rgba

    elif i[0] in blood_nodes and i[1] in blood_nodes:
       l_blood_edges.append(i)
       linkcol_blood_heart[i] = col_blood_rgba

    else:
        linkcol_blood_heart[i] = (0,0,0,0)



In [35]:
layout_name = "scene03-Organs-Bloodsystem-Heart"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_blood_heart, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_blood_heart, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_organs, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 4 - all organs

In [36]:
# node colors 

col_liver_rgba = (153,80,3,100)
col_heart_rgba = (228,9,9,100)
col_kidney_rgba = (142,0,0,100)
col_blood_rgba = (250,84,0,100) 

liver_nodes = []
for node, attr in node_attr:
    if 'Liver-' in attr['name']:
        liver_nodes.append(node)
heart_nodes = []
for node, attr in node_attr:
    if 'Heart-' in attr['name']:
        heart_nodes.append(node)
kidney_nodes = []
for node, attr in node_attr:
    if 'Kidneys-' in attr['name']:
        kidney_nodes.append(node)
blood_nodes = []
for node, attr in node_attr:
    if 'BloodSystem-' in attr['name']:
        blood_nodes.append(node)

nodecol_organs = {}
for i in G.nodes():
    if i in liver_nodes:
        nodecol_organs[i] = col_liver_rgba
    elif i in heart_nodes:
        nodecol_organs[i] = col_heart_rgba
    elif i in kidney_nodes:
        nodecol_organs[i] = col_kidney_rgba
    elif i in blood_nodes:
       nodecol_organs[i] = col_blood_rgba
    else:
        nodecol_organs[i] = (0,0,0,0)

In [37]:
pos_organs = {}
for i in G.nodes():
    if i in liver_nodes or i in heart_nodes or i in kidney_nodes or i in blood_nodes:
        pos_organs[i] = G.nodes[i]['pos']
    elif i in ppi_nodes:
        pos_organs[i] = generate_random_spherical_position()
    else:
        pos_organs[i] = generate_random_central_position()

In [38]:
# link colors 

linkcol_organs = {}

l_liver_edges = []
l_heart_edges = []
l_kidney_edges = []
l_blood_edges = []

for i in G.edges():
    if i[0] in liver_nodes and i[1] in liver_nodes:
        l_liver_edges.append(i)
        linkcol_organs[i] = col_liver_rgba
    
    elif i[0] in heart_nodes and i[1] in heart_nodes:
        l_heart_edges.append(i)
        linkcol_organs[i] = col_heart_rgba

    elif i[0] in kidney_nodes and i[1] in kidney_nodes:
        l_kidney_edges.append(i)
        linkcol_organs[i] = col_kidney_rgba

    elif i[0] in blood_nodes and i[1] in blood_nodes:
       l_blood_edges.append(i)
       linkcol_organs[i] = col_blood_rgba

    else:
        linkcol_organs[i] = (0,0,0,0)



In [39]:
layout_name = "scene04-Organs-all"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_organs, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_organs, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_organs, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 5 - BPA in center + PPI surrounding

In [40]:
bpa_nodes = []
for node, attr in node_attr:
    if 'bpa' in node_type[node]:
        bpa_nodes.append(node)
print(len(bpa_nodes))

500


In [41]:
# node positions

pos_bpa = {}
for i in G.nodes():
    if i in bpa_nodes or i in ppi_nodes:
        pos_bpa[i] = G.nodes[i]['pos']
    else:
        pos_bpa[i] = generate_random_spherical_position(spatial_range=(0.0,0.05))

In [45]:
# node colors

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib as mpl    

col_palette = cm.get_cmap('Blues', len(ppi_nodes))
norm = mpl.colors.Normalize(vmin=0, vmax=len(ppi_nodes)-1)
cmap = mpl.cm.ScalarMappable(norm=norm, cmap=col_palette)
cmap.set_array([])

d_nodecolors_cmap = {}
for i, node in enumerate(ppi_nodes):
    rgba = cmap.to_rgba(i)
    d_nodecolors_cmap[node] = (rgba[0]*255, rgba[1]*255, rgba[2]*255, 100)

    
nodecol_bpa = {}
for i in G.nodes():
    if i in bpa_nodes:
        nodecol_bpa[i] = (220,220,0,100)
    elif i in ppi_nodes:
        nodecol_bpa[i] = d_nodecolors_cmap[i]
    else:
        nodecol_bpa[i] = (0,0,0,0)

C:\Users\chris\AppData\Local\Temp\ipykernel_33256\2797485455.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  col_palette = cm.get_cmap('Blues', len(ppi_nodes))


In [46]:
# link colors

linkcol_bpa = {}
l_bpa_edges = []

for i in G.edges():
    if i[0] in bpa_nodes and i[1] in bpa_nodes:
        l_bpa_edges.append(i)
        linkcol_bpa[i] = (220,220,0,100)
    elif i[0] in ppi_nodes and i[1] in ppi_nodes:
        linkcol_bpa[i] = (0,164,215,50)
    else:
        linkcol_bpa[i] = (0,0,0,0)

In [47]:
layout_name = "scene05-BPA-molecule"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_bpa, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_bpa, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_bpa, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 6 - bpa affected genes highlighted 

In [52]:
#highlight genes in ppi which are affected by BPA 

bpa_affected_genes = []
with open ("temp-files/Microplastics/BPA_genes.txt", "r") as f:
    for line in f:
        bpa_affected_genes.append(line.strip())
    
print("bpa_affected_genes:", len(bpa_affected_genes))

bpa_affected_genes: 10401


In [53]:
d_id_nodename = {}
for node, attr in node_attr:
    d_id_nodename[node] = attr['name']

In [54]:
# node colors 

nodecol_bpa_affected = {}
for i, n in d_id_nodename.items():
    if n in bpa_affected_genes:
        nodecol_bpa_affected[i] = (255,0,0,180)
    elif i in bpa_nodes:
        nodecol_bpa_affected[i] = nodecol_bpa[i]
    elif i in ppi_nodes:
        nodecol_bpa_affected[i] = d_nodecolors_cmap[i]
    else:
        nodecol_bpa_affected[i] = (0,0,0,0)

In [55]:
layout_name = "scene06-BPA-molecule-affected-PPI-nodes"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_bpa_affected, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_bpa, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_bpa, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 7 - disease modules (cardio..)

In [165]:
import pickle
with open ('temp-files/Microplastics/BPA_diseases_sigfdr_dict.pickle', 'rb') as f:
    d_disease_genes = pickle.load(f)

for i,n in d_disease_genes.items():
    print(i, len(n))

Alzheimer's Disease 44
Liver carcinoma 43
Seizures 37
Colorectal Carcinoma 77
Familial thoracic aortic aneurysm and aortic dissection 24
Cardiomyopathies 38
Malignant neoplasm of breast 88
Bipolar Disorder 62
Obesity 71
Diabetes Mellitus, Non-Insulin-Dependent 82
Hypertensive disease 100
Liver Cirrhosis, Experimental 44
melanoma 46
Diabetes Mellitus, Experimental 62
Heart failure 21
Congestive heart failure 24
Autistic Disorder 32
Intellectual Disability 125
Myocardial Infarction 33
Schizophrenia 146
Breast Carcinoma 73
Malignant tumor of colon 15
Colorectal Neoplasms 20
Mammary Neoplasms 24
Leukemia, Myelocytic, Acute 53


In [166]:
dismod_1 = list(d_disease_genes["Heart failure"])+list(d_disease_genes["Cardiomyopathies"])
print("len dismod_1:", len(dismod_1))   

dismod_2 = list(d_disease_genes["Hypertensive disease"])
print("len dismod_2:", len(dismod_2))

dismod_3 = list(d_disease_genes["Intellectual Disability"])
print("len dismod_3:", len(dismod_3))

dismod_4 = list(d_disease_genes["Alzheimer's Disease"])+list(d_disease_genes["Seizures"])
print("len dismod_4:", len(dismod_4))

len dismod_1: 59
len dismod_2: 100
len dismod_3: 125
len dismod_4: 81


In [167]:
d_id_dismod_1 = {}
d_id_dismod_2 = {}
d_id_dismod_3 = {}
d_id_dismod_4 = {}
for k,v in d_id_nodename.items():
    if v in dismod_1:
        d_id_dismod_1[k] = v
    elif v in dismod_2:
        d_id_dismod_2[k] = v
    elif v in dismod_3:
        d_id_dismod_3[k] = v
    elif v in dismod_4:
        d_id_dismod_4[k] = v        

In [168]:
# make subgraphs 

G_dismod_1 = G.subgraph(d_id_dismod_1.keys())
print("G_dismod_1_nodes:", len(G_dismod_1.nodes()))
print("G_dismod_1_edges:", len(G_dismod_1.edges()))

G_dismod_2 = G.subgraph(d_id_dismod_2.keys())
print("G_dismod_2_nodes:", len(G_dismod_2.nodes()))
print("G_dismod_2_edges:", len(G_dismod_2.edges()))

G_dismod_3 = G.subgraph(d_id_dismod_3.keys())
print("G_dismod_3_nodes:", len(G_dismod_3.nodes()))
print("G_dismod_3_edges:", len(G_dismod_3.edges()))

G_dismod_4 = G.subgraph(d_id_dismod_4.keys())
print("G_dismod_4_nodes:", len(G_dismod_4.nodes()))
print("G_dismod_4_edges:", len(G_dismod_4.edges()))

G_dismod_1_nodes: 56
G_dismod_1_edges: 64
G_dismod_2_nodes: 86
G_dismod_2_edges: 125
G_dismod_3_nodes: 81
G_dismod_3_edges: 64
G_dismod_4_nodes: 64
G_dismod_4_edges: 42


In [ ]:
# node positions 

pos_diseases_heart = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
        pos_dis_1 = nx.spring_layout(G_dismod_1, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_1_rescaled = {node: (x*0.1+0.6, y*0.1+0.6, z*0.1+0.6) for node, (x,y,z) in pos_dis_1.items()}
        pos_diseases_heart[i] = pos_dis_1_rescaled[i]
    elif i in d_id_dismod_2.keys():
        pos_dis_2 = nx.spring_layout(G_dismod_2, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_2_rescaled = {node: (x*0.1+0.4, y*0.1+0.4, z*0.1+0.4) for node, (x,y,z) in pos_dis_2.items()}
        pos_diseases_heart[i] = pos_dis_2_rescaled[i]
    else:
        pos_diseases_heart[i] = G.nodes[i]['pos']

In [181]:
# node colors 

col_dismod_1_rgba = (195,3,3,200)   
col_dismod_2_rgba = (255,116,116,200)
col_dismod_3_rgba = (255,100,0,200)
col_dismod_4_rgba = (255,195,120,200) #(121,59,115,200)

nodecol_diseases_heart = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
         nodecol_diseases_heart[i] = col_dismod_1_rgba
    elif i in d_id_dismod_2.keys():
        nodecol_diseases_heart[i] = col_dismod_2_rgba
    elif i in bpa_nodes:
        nodecol_diseases_heart[i] = nodecol_bpa[i]
    elif i in ppi_nodes:
        nodecol_diseases_heart[i] = d_nodecolors_cmap[i]
    else:
        nodecol_diseases_heart[i] = (0,0,0,0)

In [182]:
linkcol_diseases_heart = {}
for i in G.edges():
    if i in G_dismod_1.edges():
        linkcol_diseases_heart[i] = col_dismod_1_rgba
    elif i in G_dismod_2.edges():
        linkcol_diseases_heart[i] = col_dismod_2_rgba
    else:
        linkcol_diseases_heart[i] = (0,0,0,0)

In [183]:
layout_name = "scene07-BPA-diseases"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_diseases_heart, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_diseases_heart, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_diseases_heart, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 8 - diseasese (neurological)

In [184]:
# node positions 

pos_diseases_neuro = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
        pos_diseases_neuro[i] = pos_diseases_heart[i]
    elif i in d_id_dismod_2.keys():
        pos_diseases_neuro[i] = pos_diseases_heart[i]
    elif i in d_id_dismod_3.keys():
        pos_dis_3 = nx.spring_layout(G_dismod_3, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.001)
        pos_dis_3_rescaled = {node: (x*0.1+0.2, y*0.1+0.2, z*0.1+0.6) for node, (x,y,z) in pos_dis_3.items()}
        pos_diseases_neuro[i] = pos_dis_3_rescaled[i]
    elif i in d_id_dismod_4.keys():
        pos_dis_4 = nx.spring_layout(G_dismod_4, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_4_rescaled = {node: (x*0.1+0.6, y*0.1+0.6, z*0.1+0.2) for node, (x,y,z) in pos_dis_4.items()}
        pos_diseases_neuro[i] = pos_dis_4_rescaled[i]

    else:
        pos_diseases_neuro[i] = G.nodes[i]['pos']

In [185]:

nodecol_diseases_neuro = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
         nodecol_diseases_neuro[i] = col_dismod_1_rgba
    elif i in d_id_dismod_2.keys():
        nodecol_diseases_neuro[i] = col_dismod_2_rgba
    elif i in d_id_dismod_3.keys():
        nodecol_diseases_neuro[i] = col_dismod_3_rgba
    elif i in d_id_dismod_4.keys():
        nodecol_diseases_neuro[i] = col_dismod_4_rgba
    elif i in bpa_nodes:
        nodecol_diseases_neuro[i] = nodecol_bpa[i]
    elif i in ppi_nodes:
        nodecol_diseases_neuro[i] = d_nodecolors_cmap[i]
    else:
        nodecol_diseases_neuro[i] = (0,0,0,0)

In [186]:
linkcol_diseases_neuro = {}
for i in G.edges():
    if i in G_dismod_1.edges():
        linkcol_diseases_neuro[i] = col_dismod_1_rgba
    elif i in G_dismod_2.edges():
        linkcol_diseases_neuro[i] = col_dismod_2_rgba
    elif i in G_dismod_3.edges():
        linkcol_diseases_neuro[i] = col_dismod_3_rgba
    elif i in G_dismod_4.edges():
        linkcol_diseases_neuro[i] = col_dismod_4_rgba
    # elif i[0] in ppi_nodes and i[1] in ppi_nodes:
    #     linkcol_diseases[i] = (0,164,215,50)
    else:
        linkcol_diseases_neuro[i] = (0,0,0,0)

In [188]:
layout_name = "scene08-BPA-diseases_all"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_diseases_neuro, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_diseases_neuro, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_diseases_neuro, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


✅ Connected to /main
